In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, f1_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json

# Step 1: Load and preprocess the dataset
data = pd.read_csv('./processed_heart.csv')  # Replace with the correct filename

# Separate features and target
X = data.drop(columns=['num'])  # 'num' is the target column
y = data['num']

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# Convert data to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)

# Define the neural network for classification
class ClassificationNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ClassificationNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x, weights=None, biases=None):
        if weights is not None and biases is not None:
            with torch.no_grad():
                self.fc1.weight.copy_(weights[0].detach())
                self.fc1.bias.copy_(biases[0].detach())
                self.fc2.weight.copy_(weights[1].detach())
                self.fc2.bias.copy_(biases[1].detach())
                self.fc3.weight.copy_(weights[2].detach())
                self.fc3.bias.copy_(biases[2].detach())

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.softmax(x, dim=1)  # Apply softmax to the output layer


# Helper functions for evolutionary strategy
def assign_weights_to_model(model, flat_params):
    current_pos = 0
    weights, biases = [], []
    for param in model.parameters():
        num_params = param.numel()
        reshaped_param = flat_params[current_pos:current_pos + num_params].view(param.shape)
        if len(param.shape) > 1:
            weights.append(reshaped_param)
        else:
            biases.append(reshaped_param)
        current_pos += num_params
    return weights, biases


def fitness_function(flat_params, model, X, y):
    weights, biases = assign_weights_to_model(model, flat_params)
    outputs = model(X, weights, biases)
    loss_fn = nn.CrossEntropyLoss()
    loss = loss_fn(outputs, y)

    predictions = torch.argmax(outputs, dim=1).numpy()
    y_true = y.numpy()

    precision = precision_score(y_true, predictions, average='weighted', zero_division=0)
    f1 = f1_score(y_true, predictions, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_true, predictions)

    fitness = accuracy + 0.2 * precision + 0.1 * f1 - 0.1 * loss.item()
    return fitness, accuracy, f1


def initialize_population_with_strategy_parameters(pop_size, model, lower_bound=-1.0, upper_bound=1.0):
    population, step_sizes = [], []
    for _ in range(pop_size):
        individual, individual_step_sizes = [], []
        for param in model.parameters():
            init_param = torch.empty_like(param).uniform_(lower_bound, upper_bound).flatten()
            init_step_size = torch.full_like(init_param, 0.1)  # Initial mutation step size
            individual.append(init_param)
            individual_step_sizes.append(init_step_size)
        population.append(torch.cat(individual))
        step_sizes.append(torch.cat(individual_step_sizes))
    return population, step_sizes


def adaptive_mutate_with_self_adaptation(individual, step_sizes, tau_global=0.1, tau_individual=0.1, lower_bound=-1.0, upper_bound=1.0):
    n = individual.numel()
    global_step_adjustment = torch.exp(tau_global * torch.randn(1))
    individual_step_adjustment = torch.exp(tau_individual * torch.randn(n))
    step_sizes *= global_step_adjustment * individual_step_adjustment

    noise = torch.randn_like(individual) * step_sizes
    mutated_individual = individual + noise
    mutated_individual = torch.clamp(mutated_individual, lower_bound, upper_bound)
    return mutated_individual, step_sizes


def selection(population, fitnesses, strategy, mu, lam):
    selected_indices = np.argsort(fitnesses)[-mu:]
    return [population[i] for i in selected_indices]


# Evolution Strategy Implementation
def evolution_strategy_with_updates(model, X_train, y_train, generations=50, pop_size=50, mutation_rate=0.05, strategy='(μ, λ)', lower_bound=-1.0, upper_bound=1.0):
    population, step_sizes = initialize_population_with_strategy_parameters(pop_size, model, lower_bound, upper_bound)
    history = {'fitness': [], 'accuracy': [], 'f1_score': []}
    weights_and_biases_history = []
    mu = 25
    lam = 75

    for gen in range(generations):
        fitnesses, accuracies, f1_scores = [], [], []

        for individual in population:
            fitness, accuracy, f1 = fitness_function(individual, model, X_train, y_train)
            fitnesses.append(fitness)
            accuracies.append(accuracy)
            f1_scores.append(f1)

        best_idx = np.argmax(fitnesses)
        best_weights, best_biases = assign_weights_to_model(model, population[best_idx])
        weights_and_biases_history.append({
            'generation': gen,
            'weights': [w.clone().detach().cpu().numpy().tolist() for w in best_weights],
            'biases': [b.clone().detach().cpu().numpy().tolist() for b in best_biases],
        })

        offspring_population, offspring_step_sizes = [], []
        for individual, step_size in zip(population, step_sizes):
            mutated_individual, updated_step_size = adaptive_mutate_with_self_adaptation(
                individual, step_size, tau_global=0.1, tau_individual=0.1, lower_bound=lower_bound, upper_bound=upper_bound
            )
            offspring_population.append(mutated_individual)
            offspring_step_sizes.append(updated_step_size)

        population = selection(offspring_population, fitnesses, strategy, mu, lam)
        step_sizes = [offspring_step_sizes[i] for i in range(len(population))]

        best_fitness = max(fitnesses)
        best_accuracy = max(accuracies)
        best_f1 = max(f1_scores)

        history['fitness'].append(best_fitness)
        history['accuracy'].append(best_accuracy)
        history['f1_score'].append(best_f1)

        print(f"Generation {gen + 1}: Best Fitness = {best_fitness:.4f}, Accuracy = {best_accuracy:.4f}, F1 = {best_f1:.4f}")

    with open('weights_biases_history.json', 'w') as f:
        json.dump(weights_and_biases_history, f, indent=4)

    return history


# Plotting Functions
def plot_metrics(history):
    sns.set_theme(style="darkgrid")
    generations = range(len(history['fitness']))

    plt.figure(figsize=(12, 6))
    plt.plot(generations, history['fitness'], label='Fitness', linewidth=2)
    plt.plot(generations, history['accuracy'], label='Accuracy', linestyle='--', linewidth=2)
    plt.plot(generations, history['f1_score'], label='F1 Score', linestyle='-.', linewidth=2)
    plt.xlabel('Generation')
    plt.ylabel('Metrics')
    plt.title('Metrics over Generations', fontsize=16)
    plt.legend()
    plt.show()


# Run Experiment
input_size = X_train.shape[1]
hidden_size = 5
output_size = len(np.unique(y))

model = ClassificationNN(input_size, hidden_size, output_size)
history = evolution_strategy_with_updates(model, X_train_tensor, y_train_tensor)

plot_metrics(history)
